# Notebook 3: Historical recall with session branches

This notebook demonstrates the current branch-aware recall story in the repository: each session writes memories into a session branch, and the journal preserves the sequence of changes on disk.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from datetime import date

from src.active_memory.models import WorkingItem
from src.persistence.pipeline import MutationPipeline
from src.terminus.adapter import TerminusMemoryRepository
from src.terminus.branch_manager import session_branch_name

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-03"
shutil.rmtree(memory_root, ignore_errors=True)
# This intentionally points at an unused local address so the repository
# uses its in-process fallback store without requiring a live TerminusDB server.
repo = TerminusMemoryRepository(url="http://localhost:9999")
pipeline = MutationPipeline(memory_root, enable_terminus=True, terminus_repo=repo)


In [ ]:
pipeline.run(
    WorkingItem(
        item_type="observation",
        content="During the morning deploy, Acme Payments rotated the edge certificate for checkout traffic.",
        session_id="history-a",
    ),
    session_id="history-a",
)
pipeline.run(
    WorkingItem(
        item_type="observation",
        content="Later, Acme Payments restored the previous certificate after customers reported checkout failures.",
        session_id="history-b",
    ),
    session_id="history-b",
)
show({
    "history_a_branch": session_branch_name("history-a"),
    "history_b_branch": session_branch_name("history-b"),
})

In [ ]:
history_a_memories = repo.query_memories(branch=session_branch_name("history-a"))
history_b_memories = repo.query_memories(branch=session_branch_name("history-b"))
show({
    "history_a_memories": history_a_memories,
    "history_b_memories": history_b_memories,
})

In [ ]:
journal_file = memory_root / "journal" / f"{date.today().isoformat()}.jsonl"
journal_rows = [json.loads(line) for line in journal_file.read_text().splitlines() if line.strip()]
show({
    "journal_file": str(journal_file),
    "events_recorded": len(journal_rows),
    "events": journal_rows,
})

## What this notebook demonstrates

- branch-local memory for separate sessions
- explicit recall of prior session memories by branch name
- journal-backed chronology of memory mutations
- a simple historical recall path without requiring a live TerminusDB server